# **Project Name** - Uber Supply–Demand Gap Analysis

##### **Project Type** - Exploratory Data Analysis (EDA)
##### **Contribution** - Individual
##### **Team Member 1** - Aditya Kumar Yadav

# **Project Summary -**

This project analyses Uber request records to understand the supply–demand gap between passengers requesting rides and drivers available to fulfil them. The dataset contains 6,745 requests and six original fields: Request id, Pickup point, Driver id, Status, Request timestamp, and Drop timestamp. Each row represents one ride request. The outcome field has three categories: Trip Completed, Cancelled, and No Cars Available. The analysis combines data inspection, cleaning, feature engineering, exploratory visualisation, and business recommendations.

The first step is to inspect the data structure, column types, missing values, and duplicate records. The supplied CSV represents unavailable driver IDs and drop timestamps as missing values. These are meaningful in context: a missing driver ID is expected when no driver was assigned, and a missing drop timestamp is expected when a trip did not complete. Therefore, the analysis does not automatically remove those rows. Instead, it preserves the request-level record and treats the missingness as part of the service outcome. Request timestamps are converted to datetime values so that hour, date, day of week, and broad time slot can be derived. Trip duration is calculated only when both timestamps are available and the drop time is not earlier than the request time.

The main outcome counts are 2,831 completed trips, 1,264 cancellations, and 2,650 requests with no cars available. In percentage terms, approximately 42.0% of requests completed, 18.7% were cancelled, and 39.3% had no cars available. Thus, around 58% of requests did not result in a completed trip in this dataset. This indicates a substantial service reliability issue and provides a clear reason to examine the timing and location of unsuccessful requests rather than looking only at total demand.

The analysis compares outcomes by pickup point (City versus Airport), hour of day, and broader time slot (Early Morning, Morning, Afternoon, Evening, and Night). It uses frequency charts, stacked comparisons, hourly trends, rate comparisons, a correlation heatmap, and a pair plot to explore patterns. The hourly and time-slot analysis is particularly important because driver supply may not be evenly distributed through the day. The status-by-pickup comparison helps distinguish driver cancellations from a lack of available cars. The request-hour charts also provide a practical view of when demand is highest and when successful trips fall behind total requests.

The business objective is to identify actionable periods and locations for improving supply coverage. Recommendations should be linked to measured patterns: schedule more drivers in time slots with a high number or share of no-car requests; consider targeted incentives for drivers serving under-covered routes; and investigate cancellation causes before changing driver policies. A pilot programme should compare completion rate, cancellation rate, no-car rate, and driver utilisation before and after changes. Any surge pricing or incentive strategy should be evaluated for fairness, customer affordability, and driver participation.

This project is descriptive rather than causal. The dataset does not include fare, driver shift, travel distance, weather, traffic, or the reasons behind cancellations. Consequently, it can identify associations and operational pressure points, but cannot prove why each request failed. The final recommendations are therefore framed as testable operational actions. The deliverables include an Excel workbook with cleaned data and a dashboard, a MySQL script with reusable analytical queries, this EDA notebook, and a findings report. Together, these outputs provide a reproducible view of request outcomes and a practical starting point for improving Uber's ride fulfilment.

# **GitHub Link -**

https://github.com/adityayadavkumar/Uber-Demand-Gap-Analysis/blob/main/Uber_Supply_Demand_Gap_EDA.ipynb

# **Problem Statement**

Ride requests can fail because the assigned driver cancels or no car is available. This project identifies the time slots and pickup points associated with these outcomes.

#### **Define Your Business Objective?**

Quantify request outcomes, identify when unsuccessful requests are concentrated, and recommend evidence-based changes to improve driver availability and completion rates.

# **General Guidelines**

- Run the notebook from top to bottom.
- Keep the CSV in the same folder as this notebook or update `DATA_PATH`.
- Missing driver IDs and drop timestamps are meaningful for unsuccessful requests and should not be blindly deleted.
- Charts describe associations; they do not prove causation.

# ***Let's Begin!***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
DATA_PATH = Path("Uber Request Data.csv")
# If running from another folder, set DATA_PATH to the full path of the CSV.


### Dataset Loading

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
df = df_raw.copy()
df.head()


### Dataset First View

In [ ]:
display(df.head(10))
print("Shape:", df.shape)


### Dataset Rows & Columns count

In [ ]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")


### Dataset Information

In [ ]:
df.info()


#### Duplicate Values

In [ ]:
print("Duplicate rows:", df.duplicated().sum())


#### Missing Values/Null Values

In [ ]:
missing_summary = pd.DataFrame({
    "Missing Count": df.isna().sum(),
    "Missing %": (df.isna().mean() * 100).round(2)
})
display(missing_summary)


In [ ]:
plt.figure(figsize=(9, 4))
sns.heatmap(df.isna(), cbar=False, yticklabels=False)
plt.title("Missing-Value Pattern")
plt.xlabel("Columns")
plt.tight_layout()
plt.show()


### What did you know about your dataset?

The source has 6,745 rows and six columns. Status has three outcomes. Missing Driver id and Drop timestamp values are structurally meaningful: unsuccessful requests often have no assigned driver or completed-trip timestamp. The raw dataset should be retained, and missing values should be interpreted in context rather than automatically dropped.

## ***2. Understanding Your Variables***

In [ ]:
display(df.columns.to_frame(index=False, name="Column"))
display(df.describe(include="all").T)


### Variables Description

- **Request id:** unique identifier for each request.
- **Pickup point:** City or Airport.
- **Driver id:** assigned driver identifier; missing where no driver was assigned.
- **Status:** Trip Completed, Cancelled, or No Cars Available.
- **Request timestamp:** time when the request was made.
- **Drop timestamp:** completion/drop-off time; missing for unsuccessful requests.

### Check Unique Values for each variable

In [ ]:
for col in df.columns:
    print(f"{col}: {df[col].nunique(dropna=True)} unique non-null values")
    if df[col].nunique(dropna=True) <= 15:
        print(df[col].value_counts(dropna=False), "\\n")


## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
# Parse timestamps, preserving rows with missing values.
df["Request timestamp"] = pd.to_datetime(df["Request timestamp"], dayfirst=True, errors="coerce")
df["Drop timestamp"] = pd.to_datetime(df["Drop timestamp"], dayfirst=True, errors="coerce")
df["Driver id"] = pd.to_numeric(df["Driver id"], errors="coerce")

df["Request hour"] = df["Request timestamp"].dt.hour
df["Request date"] = df["Request timestamp"].dt.date
df["Request day"] = df["Request timestamp"].dt.day_name()

def get_time_slot(hour):
    if pd.isna(hour): return "Unknown"
    hour = int(hour)
    if hour < 5: return "Early Morning"
    if hour < 12: return "Morning"
    if hour < 17: return "Afternoon"
    if hour < 21: return "Evening"
    return "Night"

df["Time slot"] = df["Request hour"].map(get_time_slot)
df["Trip duration (minutes)"] = (
    df["Drop timestamp"] - df["Request timestamp"]
).dt.total_seconds() / 60
df.loc[df["Trip duration (minutes)"] < 0, "Trip duration (minutes)"] = np.nan
df["Trip duration (minutes)"] = df["Trip duration (minutes)"].round(2)

print("Cleaned/engineered shape:", df.shape)
display(df.head())


### What all manipulations have you done and insights you found?

Timestamps are parsed as datetime values; request hour, date, day, and time slot are derived; trip duration is computed only where both timestamps exist; invalid negative durations are set to missing. No request rows are removed because failed requests are central to the business question.

## ***4. Data Visualization, Storytelling & Experimenting with Charts***

The next 20 charts investigate status mix, demand timing, pickup point differences, failed-request patterns, trip duration, and relationships among numeric variables. Each chart is paired with a short interpretation.

#### Chart 1 - Requests by Status

In [ ]:
status_counts = df["Status"].value_counts()
status_counts.plot(kind="bar", title="Requests by Status", ylabel="Requests", xlabel="Status", rot=15)
plt.tight_layout(); plt.show()

##### 1. Why did you pick the specific chart?

A bar chart makes the distribution of the three outcomes easy to compare.

##### 2. What insight(s) are found?

This shows the baseline share of completed and unsuccessful requests.

##### 3. Business impact

Use the largest unsuccessful category to prioritise operational investigation; a high failure share can damage reliability.

#### Chart 2 - Requests by Pickup Point

In [ ]:
df["Pickup point"].value_counts().plot(kind="bar", title="Requests by Pickup Point", ylabel="Requests", xlabel="Pickup Point", rot=0)
plt.tight_layout(); plt.show()

##### 1. Why did you pick the specific chart?

A bar chart compares total demand at City and Airport pickup points.

##### 2. What insight(s) are found?

This shows where requests are concentrated.

##### 3. Business impact

Plan coverage by pickup location, but pair demand counts with failure rates before reallocating drivers.

#### Chart 3 - Status by Pickup Point

In [ ]:
pd.crosstab(df["Pickup point"], df["Status"]).plot(kind="bar", stacked=True, title="Status by Pickup Point", ylabel="Requests", rot=0)
plt.tight_layout(); plt.show()

##### 1. Why did you pick the specific chart?

A stacked bar chart separates completed, cancelled, and no-car requests by pickup point.

##### 2. What insight(s) are found?

This shows whether the outcome mix differs between City and Airport.

##### 3. Business impact

Location-specific dispatch or driver allocation can be tested where unsuccessful outcomes are concentrated.

#### Chart 4 - Requests by Hour

In [ ]:
df["Request hour"].value_counts().sort_index().reindex(range(24), fill_value=0).plot(kind="line", marker="o", title="Requests by Hour", xlabel="Hour", ylabel="Requests")
plt.xticks(range(24)); plt.tight_layout(); plt.show()

##### 1. Why did you pick the specific chart?

A line chart shows hourly demand across a 24-hour day.

##### 2. What insight(s) are found?

The peak hour is the hour with the highest request count.

##### 3. Business impact

Use hourly demand to schedule shifts and evaluate whether driver supply matches demand.

#### Chart 5 - Status by Hour

In [ ]:
pd.crosstab(df["Request hour"], df["Status"]).reindex(range(24), fill_value=0).plot(kind="line", marker="o", title="Status by Hour", xlabel="Hour", ylabel="Requests")
plt.xticks(range(24)); plt.tight_layout(); plt.show()

##### 1. Why did you pick the specific chart?

A multi-line chart compares each outcome by request hour.

##### 2. What insight(s) are found?

The chart identifies hours when cancellations or no-car requests rise.

##### 3. Business impact

Target the relevant hour with a pilot staffing or incentive plan, then measure change.

#### Chart 6 - Status by Time Slot

In [ ]:
slot_order = ["Early Morning","Morning","Afternoon","Evening","Night"]
pd.crosstab(df["Time slot"], df["Status"]).reindex(slot_order, fill_value=0).plot(kind="bar", stacked=True, title="Status by Time Slot", ylabel="Requests", rot=20)
plt.tight_layout(); plt.show()

##### 1. Why did you pick the specific chart?

A stacked bar chart compares outcomes across five time slots.

##### 2. What insight(s) are found?

This provides a simple summary of when requests succeed or fail.

##### 3. Business impact

Use time-slot trends for shift planning, while retaining the hourly view for precision.

#### Chart 7 - No Cars Available by Time Slot

In [ ]:
df.loc[df["Status"]=="No Cars Available","Time slot"].value_counts().reindex(slot_order, fill_value=0).plot(kind="bar", title="No Cars Available by Time Slot", ylabel="Requests", rot=20)
plt.tight_layout(); plt.show()

##### 1. Why did you pick the specific chart?

A bar chart isolates requests with no available car.

##### 2. What insight(s) are found?

The highest bar identifies the slot with the largest count of unmet requests.

##### 3. Business impact

Add driver availability or shift coverage in high-pressure slots and measure no-car rate.

#### Chart 8 - Cancellations by Time Slot

In [ ]:
df.loc[df["Status"]=="Cancelled","Time slot"].value_counts().reindex(slot_order, fill_value=0).plot(kind="bar", title="Cancellations by Time Slot", ylabel="Requests", rot=20)
plt.tight_layout(); plt.show()

##### 1. Why did you pick the specific chart?

A bar chart isolates cancelled requests by time slot.

##### 2. What insight(s) are found?

The highest bar identifies where cancellations are most frequent by count.

##### 3. Business impact

Investigate reasons for cancellation before changing policies or offering incentives.

#### Chart 9 - Cancellation Rate by Pickup Point

In [ ]:
totals = df.groupby("Pickup point").size()
rates = df[df["Status"]=="Cancelled"].groupby("Pickup point").size().div(totals).fillna(0)*100
rates.plot(kind="bar", title="Cancellation Rate by Pickup Point", ylabel="Cancellation rate (%)", rot=0)
plt.tight_layout(); plt.show()

##### 1. Why did you pick the specific chart?

A rate comparison accounts for different request totals at each pickup point.

##### 2. What insight(s) are found?

Compare percentages rather than raw counts to avoid volume-only conclusions.

##### 3. Business impact

Pilot location-specific improvements where cancellation rate is higher.

#### Chart 10 - No-Car Rate by Pickup Point

In [ ]:
totals = df.groupby("Pickup point").size()
rates = df[df["Status"]=="No Cars Available"].groupby("Pickup point").size().div(totals).fillna(0)*100
rates.plot(kind="bar", title="No-Car Rate by Pickup Point", ylabel="No-car rate (%)", rot=0)
plt.tight_layout(); plt.show()

##### 1. Why did you pick the specific chart?

A rate comparison shows the proportion of requests with no car available at each pickup point.

##### 2. What insight(s) are found?

This indicates which pickup point has a greater relative availability problem.

##### 3. Business impact

Use rate and count together when deciding where to add supply.

#### Chart 11 - Completed Trips by Hour

In [ ]:
df.loc[df["Status"]=="Trip Completed","Request hour"].value_counts().sort_index().reindex(range(24), fill_value=0).plot(kind="line", marker="o", title="Completed Trips by Hour", xlabel="Hour", ylabel="Completed trips")
plt.xticks(range(24)); plt.tight_layout(); plt.show()

##### 1. Why did you pick the specific chart?

A line chart shows successful trip volume by hour.

##### 2. What insight(s) are found?

This shows when the system converts requests into completed trips.

##### 3. Business impact

Compare completed trips with total requests to identify opportunity gaps.

#### Chart 12 - Completed Rate by Hour

In [ ]:
ct = pd.crosstab(df["Request hour"], df["Status"]).reindex(range(24), fill_value=0)
rate = (ct.get("Trip Completed", 0) / ct.sum(axis=1).replace(0, np.nan))*100
rate.plot(kind="line", marker="o", title="Completion Rate by Hour", xlabel="Hour", ylabel="Completion rate (%)")
plt.xticks(range(24)); plt.tight_layout(); plt.show()

##### 1. Why did you pick the specific chart?

A line chart shows completed trips divided by requests for each hour.

##### 2. What insight(s) are found?

This reveals whether the probability of completion changes through the day.

##### 3. Business impact

Use this as a service-level measure alongside absolute counts.

#### Chart 13 - Requests by Day of Week

In [ ]:
df["Request day"].value_counts().reindex(["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]).plot(kind="bar", title="Requests by Day of Week", ylabel="Requests", rot=25)
plt.tight_layout(); plt.show()

##### 1. Why did you pick the specific chart?

A bar chart compares demand across weekdays.

##### 2. What insight(s) are found?

This identifies any day-of-week variation in request volume.

##### 3. Business impact

Use day-of-week patterns for scheduling only if the pattern is operationally consistent.

#### Chart 14 - Status by Day of Week

In [ ]:
pd.crosstab(df["Request day"], df["Status"]).reindex(["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]).plot(kind="bar", stacked=True, title="Status by Day of Week", ylabel="Requests", rot=25)
plt.tight_layout(); plt.show()

##### 1. Why did you pick the specific chart?

A stacked chart compares outcomes across weekdays.

##### 2. What insight(s) are found?

This shows whether failure mix varies by day.

##### 3. Business impact

Investigate recurring patterns; do not infer causes from day labels alone.

#### Chart 15 - Trip Duration Distribution

In [ ]:
sns.histplot(data=df.loc[df["Status"]=="Trip Completed"], x="Trip duration (minutes)", bins=30, kde=True)
plt.title("Trip Duration Distribution (Completed Trips)"); plt.tight_layout(); plt.show()

##### 1. Why did you pick the specific chart?

A histogram summarises durations for completed trips only.

##### 2. What insight(s) are found?

The distribution shows typical trip durations and outliers.

##### 3. Business impact

Use robust summaries and investigate extreme values; duration alone does not measure service quality.

#### Chart 16 - Trip Duration by Pickup Point

In [ ]:
sns.boxplot(data=df.loc[df["Status"]=="Trip Completed"], x="Pickup point", y="Trip duration (minutes)")
plt.title("Trip Duration by Pickup Point"); plt.tight_layout(); plt.show()

##### 1. Why did you pick the specific chart?

A box plot compares completed-trip duration across pickup points.

##### 2. What insight(s) are found?

This highlights differences in medians and spread.

##### 3. Business impact

Consider route context before interpreting duration differences as performance differences.

#### Chart 17 - Request Hour by Status (Box Plot)

In [ ]:
sns.boxplot(data=df, x="Status", y="Request hour")
plt.title("Request Hour by Status"); plt.xticks(rotation=15); plt.tight_layout(); plt.show()

##### 1. Why did you pick the specific chart?

A box plot compares request hour by outcome.

##### 2. What insight(s) are found?

Different hour distributions can show when each status tends to occur.

##### 3. Business impact

Combine with count-based charts because box plots do not show absolute demand.

#### Chart 18 - Requests by Hour and Pickup Point

In [ ]:
pd.crosstab(df["Request hour"], df["Pickup point"]).reindex(range(24), fill_value=0).plot(kind="line", marker="o", title="Requests by Hour and Pickup Point", xlabel="Hour", ylabel="Requests")
plt.xticks(range(24)); plt.tight_layout(); plt.show()

##### 1. Why did you pick the specific chart?

A two-line chart compares hourly City and Airport demand.

##### 2. What insight(s) are found?

This identifies when demand at each pickup point rises.

##### 3. Business impact

Use separate shift coverage where the hourly profiles differ.

#### Chart 19 - Correlation Heatmap

In [ ]:
numeric = df[["Request hour","Trip duration (minutes)"]].copy()
numeric["Status code"] = df["Status"].map({"Cancelled":0, "No Cars Available":1, "Trip Completed":2})
sns.heatmap(numeric.corr(numeric_only=True), annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Correlation Heatmap"); plt.tight_layout(); plt.show()

##### 1. Why did you pick the specific chart?

A heatmap summarises pairwise correlations among numeric engineered variables.

##### 2. What insight(s) are found?

Correlations are descriptive and can be weak or misleading when numeric status codes are used.

##### 3. Business impact

Do not treat correlation as proof of cause; use it to guide follow-up analysis.

#### Chart 20 - Pair Plot

In [ ]:
pair_data = df[["Request hour","Trip duration (minutes)","Pickup point","Status"]].dropna().sample(n=min(500, df[["Request hour","Trip duration (minutes)","Pickup point","Status"]].dropna().shape[0]), random_state=42)
sns.pairplot(pair_data, vars=["Request hour","Trip duration (minutes)"], hue="Status", corner=True, plot_kws={"alpha":0.5, "s":18})
plt.show()

##### 1. Why did you pick the specific chart?

A pair plot shows pairwise patterns among a sample of numeric variables.

##### 2. What insight(s) are found?

It provides a compact view of numeric relationships and possible outliers.

##### 3. Business impact

Use only as an exploratory aid; this dataset has limited numeric predictors.

## **5. Solution to Business Objective**

1. **Improve driver coverage in high no-car periods:** use the no-car count and rate by hour/time slot to place more drivers where requests are most often unfulfilled.
2. **Address cancellations separately from availability:** cancellation and no-car outcomes are different problems. Review cancellation reasons, route preferences, and driver/customer communications before choosing an intervention.
3. **Pilot targeted incentives:** if an under-covered period or pickup point is confirmed, test limited driver incentives and compare completion rate, no-car rate, cancellations, and driver utilisation before and after.
4. **Measure outcomes:** track weekly by hour and pickup point; avoid relying on total request counts alone.
5. **Acknowledge limitations:** the dataset has no fare, traffic, weather, driver schedule, or cancellation-reason fields, so it cannot explain every failure or establish causality.

# **Conclusion**

The dataset contains 6,745 requests, with 2,831 completed trips (42.0%), 1,264 cancellations (18.7%), and 2,650 no-car requests (39.3%). This means about 58.0% of requests did not become completed trips. The most useful next step is to target operational actions using hourly and pickup-point outcome rates, then validate impact with a pilot. Missing driver and drop timestamps are meaningful, not reasons to discard the records.

### ***Project EDA completed — remember to run all cells and save the executed notebook.***